# 02 · Error bars, clustered errors and paired comparisons

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/evals/02_error_bars.ipynb) &nbsp; 📖 Concept deck: [Error bars and comparing models](https://nilesh-salpe.github.io/visual-ai/evals/04-error-bars/) &nbsp; ⏱️ Suggested time: **40 min**

**You will build:** the statistics an eval report needs: the standard error of a mean score, clustered standard errors for grouped questions, a paired test for comparing two models, and a sample-size calculation.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def simulate_eval(n_q=1000, cluster_size=1, gap=0.0, seed=0, base=0.7, share=0.75):
    """Two models answer the same questions. Questions come in clusters that share difficulty.
    Returns (scores_a, scores_b, cluster_ids) with 0/1 scores."""
    rng = np.random.default_rng(seed)
    n_c = n_q // cluster_size
    cl_diff = rng.normal(0, 1.5, n_c)                       # cluster difficulty (shared within cluster)
    q_diff = np.repeat(cl_diff, cluster_size) + rng.normal(0, 1.0, n_c * cluster_size)
    logit = np.log(base / (1 - base)) - q_diff
    pa = sigmoid(logit); pb = sigmoid(logit + gap)
    u = rng.random(len(pa))                                 # partly shared randomness: the models tend to fail the same questions
    u_b = np.where(rng.random(len(pa)) < share, u, rng.random(len(pa)))
    return (u < pa).astype(float), (u_b < pb).astype(float), np.repeat(np.arange(n_c), cluster_size)

## Task 1: Standard error of the mean score

An eval score is a sample mean over questions, so it has sampling error. By the central limit theorem,
$$\mathrm{SE}(\hat\mu) = \sqrt{\frac{1}{n}\cdot\frac{1}{n-1}\sum_i (s_i - \hat\mu)^2}$$
and a 95% confidence interval is $\hat\mu \pm 1.96\,\mathrm{SE}$. Miller (2024), "Adding Error Bars to Evals", recommends this over bootstrapping as the default. Return the SE.

In [ ]:
def standard_error(scores):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_standard_error(f):
    s = np.array([1, 0, 1, 1, 0, 1, 1, 1.0])
    assert np.isclose(f(s), s.std(ddof=1) / np.sqrt(len(s)))
    a, _, _ = simulate_eval(4000)
    assert 0.006 < f(a) < 0.009, f"for ~70% accuracy on 4000 questions SE should be ~0.007, got {f(a):.4f}"

check("standard_error", _test_standard_error, standard_error)

## Task 2: Clustered standard errors

Many evals contain **groups** of related questions: several questions about one passage (DROP, RACE), the same problem in many languages (MGSM). Their errors are correlated, so the effective sample size is smaller than n. The clustered SE sums residuals within each cluster first:
$$\mathrm{SE}_{clustered} = \frac{1}{n}\sqrt{\sum_{c}\Big(\sum_{i \in c}(s_i - \hat\mu)\Big)^2}$$
With one question per cluster this reduces to the naive SE (up to the n vs n−1 factor). Miller reports clustered SEs "can be over 3X larger than naive standard errors" on real evals.

In [ ]:
def clustered_se(scores, clusters):
    # TODO: residuals, sum per cluster (np.bincount with weights), then the formula above
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_clustered_se(f):
    r = np.random.default_rng(1); s = r.integers(0, 2, 500).astype(float)
    assert np.isclose(f(s, np.arange(500)), np.sqrt(((s - s.mean()) ** 2).sum()) / 500), "singleton clusters = naive (ddof 0)"
    a, _, cl = simulate_eval(2000, cluster_size=10)
    assert f(a, cl) > 1.4 * a.std(ddof=1) / np.sqrt(len(a)), "with correlated clusters of 10 the clustered SE should be much larger"

check("clustered_se", _test_clustered_se, clustered_se)

## Task 3: Compare two models on the same questions

To compare models A and B on the **same** questions, don't compare two independent confidence intervals. Look at the per-question differences $d_i = s^A_i - s^B_i$:
$$\hat\delta = \bar d,\qquad \mathrm{SE}(\hat\delta) = \frac{\mathrm{sd}(d)}{\sqrt n},\qquad z = \hat\delta / \mathrm{SE}$$
Because both models find the same questions easy or hard, the differences have much less variance than the scores. Miller: conduct "statistical inference on the question-level paired differences, rather than the population-level summary statistics". Return `(delta, se, z)`.

In [ ]:
def paired_difference(a, b):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_paired_difference(f):
    a, b, _ = simulate_eval(1000, gap=0.2, seed=0)
    d, se, z = f(b, a)
    assert np.isclose(d, b.mean() - a.mean()) and np.isclose(se, (b - a).std(ddof=1) / np.sqrt(1000))
    unpaired_se = np.sqrt(a.var(ddof=1) / 1000 + b.var(ddof=1) / 1000)
    assert se < 0.75 * unpaired_se, "paired SE should be much smaller than the unpaired SE here"
    assert z > 1.96 and d / unpaired_se < 1.96, f"the paired test should detect the gap (z={z:.2f}) while comparing separate CIs does not ({d / unpaired_se:.2f})"

check("paired_difference", _test_paired_difference, paired_difference)

## Task 4: How many questions do you need?

Power analysis: to detect a true difference δ with a two-sided 5% test and 80% power, you need
$$n = \left(\frac{(z_{0.975} + z_{0.8})\,\sigma_d}{\delta}\right)^2,\qquad z_{0.975} = 1.96,\ z_{0.8} = 0.8416$$
where σ_d is the standard deviation of the per-question differences. Return n rounded **up**.

In [ ]:
def questions_needed(delta, sd_diff, z_alpha=1.959964, z_power=0.841621):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 4: run after implementing
def _test_questions_needed(f):
    want = lambda d, sd: int(np.ceil(((1.959964 + 0.841621) * sd / d) ** 2))
    assert f(0.02, 0.3) == want(0.02, 0.3), f"got {f(0.02, 0.3)}, expected {want(0.02, 0.3)}"
    assert abs(f(0.01, 0.3) / f(0.02, 0.3) - 4) < 0.01, "halving delta should quadruple n"
    assert isinstance(f(0.05, 0.5), (int, np.integer)), "return an integer (rounded up)"

check("questions_needed", _test_questions_needed, questions_needed)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: Standard error of the mean score
def standard_error_solution(scores):
    s = np.asarray(scores, float)
    return s.std(ddof=1) / np.sqrt(len(s))

check("standard_error (reference)", _test_standard_error, standard_error_solution)

In [ ]:
# @title Solution for Task 2: Clustered standard errors
def clustered_se_solution(scores, clusters):
    s = np.asarray(scores, float); e = s - s.mean()
    per_cluster = np.bincount(clusters, weights=e)
    return np.sqrt((per_cluster ** 2).sum()) / len(s)

check("clustered_se (reference)", _test_clustered_se, clustered_se_solution)

In [ ]:
# @title Solution for Task 3: Compare two models on the same questions
def paired_difference_solution(a, b):
    d = np.asarray(a, float) - np.asarray(b, float)
    delta, se = d.mean(), d.std(ddof=1) / np.sqrt(len(d))
    return delta, se, delta / se

check("paired_difference (reference)", _test_paired_difference, paired_difference_solution)

In [ ]:
# @title Solution for Task 4: How many questions do you need?
def questions_needed_solution(delta, sd_diff, z_alpha=1.959964, z_power=0.841621):
    return int(np.ceil(((z_alpha + z_power) * sd_diff / delta) ** 2))

check("questions_needed (reference)", _test_questions_needed, questions_needed_solution)

---
## Experiments
**How wide are typical error bars, and how much do clusters matter?** First the 95% CI half-width at 70% accuracy for common eval sizes. Then one simulated eval reported naively vs clustered, and finally how often a 2-point gap is detected by paired vs unpaired comparison across 500 simulated evals.

In [ ]:
for n in [100, 500, 1000, 5000, 15000]:
    print(f"n={n:6,}: 95% CI ≈ ±{1.96 * np.sqrt(0.7 * 0.3 / n) * 100:.1f} points")
a, b, cl = simulate_eval(2500, cluster_size=10, seed=2)
print(f"\nclusters of 10: naive SE {standard_error_solution(a) * 100:.2f} pts, clustered SE {clustered_se_solution(a, cl) * 100:.2f} pts "
      f"(ratio {clustered_se_solution(a, cl) / standard_error_solution(a):.2f})")
hits_p = hits_u = 0; gaps = []
for s in range(500):
    a, b, _ = simulate_eval(1000, gap=0.2, seed=100 + s)
    d, se, z = paired_difference_solution(b, a); gaps.append(d)
    hits_p += abs(z) > 1.96
    hits_u += abs(d) / np.sqrt(a.var(ddof=1) / 1000 + b.var(ddof=1) / 1000) > 1.96
print(f"\naverage gap {np.mean(gaps) * 100:.1f} pts on 1,000 questions: detected by paired test {hits_p / 5:.0f}% of the time, by unpaired {hits_u / 5:.0f}%")

---
## Takeaways
* Always report a standard error: with 1,000 questions at 70% accuracy, the 95% interval is about ±2.8 points, wider than many leaderboard gaps.
* Grouped questions need clustered standard errors. Ignoring the clustering can understate uncertainty by 3× or more.
* When comparing two models on the same questions, test the paired per-question differences: it can detect gaps that overlapping confidence intervals hide.
* Use power analysis before building an eval: the questions needed grow with $1/\delta^2$.